# M3 - Hiểu Train / Dev / Test của ViHOS

## Mục tiêu

- Kiểm tra đúng số lượng Train / Dev / Test.
- Kiểm tra Dev ở bản Syllable, Word và Span Extraction có khớp nhau hay không.
- Hiểu rõ vai trò của từng tập:
  - **Train**: dùng để model học.
  - **Dev**: dùng để thử nghiệm và điều chỉnh trong quá trình phát triển.
  - **Test**: chỉ dùng để đánh giá cuối cùng.
- Xác nhận file Test chính thức là `test/test.csv`.
- Không xem hoặc phân tích nhãn Test ở giai đoạn này.

> Quy tắc của đồ án: dù Test public có sẵn đáp án, ta coi Test là **đề thi cuối** và không dùng đáp án Test để thiết kế hay điều chỉnh model.


In [1]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "DOCS" else current_dir
data_root = project_root / "DATA" / "ViHOS_official"

# Span Extraction
train_span_path = data_root / "train_span_extraction" / "train.csv"
dev_span_path = data_root / "train_span_extraction" / "dev.csv"

# Sequence Labeling - Dev
dev_syllable_path = (
    data_root
    / "train_sequence_labeling"
    / "syllable"
    / "dev_BIO_syllable.csv"
)

dev_word_path = (
    data_root
    / "train_sequence_labeling"
    / "word"
    / "dev_BIO_Word.csv"
)

# Test chính thức
official_test_path = data_root / "test" / "test.csv"

paths = {
    "Train Span": train_span_path,
    "Dev Span": dev_span_path,
    "Dev Syllable": dev_syllable_path,
    "Dev Word": dev_word_path,
    "Official Test": official_test_path,
}

for name, path in paths.items():
    print(f"{name:<15}: {path.exists()}  |  {path}")


Train Span     : True  |  c:\cs221-FinalTerm-ViHOS\DATA\ViHOS_official\train_span_extraction\train.csv
Dev Span       : True  |  c:\cs221-FinalTerm-ViHOS\DATA\ViHOS_official\train_span_extraction\dev.csv
Dev Syllable   : True  |  c:\cs221-FinalTerm-ViHOS\DATA\ViHOS_official\train_sequence_labeling\syllable\dev_BIO_syllable.csv
Dev Word       : True  |  c:\cs221-FinalTerm-ViHOS\DATA\ViHOS_official\train_sequence_labeling\word\dev_BIO_Word.csv
Official Test  : True  |  c:\cs221-FinalTerm-ViHOS\DATA\ViHOS_official\test\test.csv


In [2]:
# Đọc Train và Dev đầy đủ.
# Với Test, chỉ đọc cột content để không đụng tới đáp án index_spans.

df_train = pd.read_csv(train_span_path)
df_dev = pd.read_csv(dev_span_path)

df_dev_syllable = pd.read_csv(dev_syllable_path)
df_dev_word = pd.read_csv(dev_word_path)

df_test = pd.read_csv(
    official_test_path,
    usecols=["content"]
)

print("Đã đọc dữ liệu.")


Đã đọc dữ liệu.


In [3]:
# Kiểm tra số lượng mẫu của 3 tập

split_sizes = pd.DataFrame({
    "Tập": ["Train", "Dev", "Test"],
    "Số comment": [
        len(df_train),
        len(df_dev),
        len(df_test),
    ],
    "Mục đích": [
        "Model học",
        "Thử nghiệm / điều chỉnh",
        "Đánh giá cuối cùng",
    ],
})

split_sizes


,Tập,Số comment,Mục đích
0,Train,8844,Model học
1,Dev,1106,Thử nghiệm / điều chỉnh
2,Test,1106,Đánh giá cuối cùng


In [4]:
# Kiểm tra số lượng đúng theo ViHOS

assert len(df_train) == 8844, f"Train khác kỳ vọng: {len(df_train)}"
assert len(df_dev) == 1106, f"Dev khác kỳ vọng: {len(df_dev)}"
assert len(df_test) == 1106, f"Test khác kỳ vọng: {len(df_test)}"

print("PASS: Train / Dev / Test có đúng kích thước 8844 / 1106 / 1106.")


PASS: Train / Dev / Test có đúng kích thước 8844 / 1106 / 1106.


In [5]:
# Kiểm tra ba phiên bản của Dev có cùng số comment

dev_syllable_comments = df_dev_syllable["sentence_id"].nunique()
dev_word_comments = df_dev_word["sentence_id"].nunique()
dev_span_comments = len(df_dev)

dev_alignment = pd.DataFrame({
    "Phiên bản Dev": [
        "Syllable",
        "Word",
        "Span Extraction",
    ],
    "Số comment": [
        dev_syllable_comments,
        dev_word_comments,
        dev_span_comments,
    ],
})

display(dev_alignment)

assert dev_syllable_comments == dev_word_comments == dev_span_comments == 1106

print("PASS: 3 phiên bản Dev khớp nhau về số comment.")


,Phiên bản Dev,Số comment
0,Syllable,1106
1,Word,1106
2,Span Extraction,1106


PASS: 3 phiên bản Dev khớp nhau về số comment.


In [6]:
# Chỉ kiểm tra cấu trúc cột.
# Không in nội dung Test và không đọc index_spans của Test.

print("Cột Train:", df_train.columns.tolist())
print("Cột Dev:", df_dev.columns.tolist())
print("Cột Test đang sử dụng trong M3:", df_test.columns.tolist())


Cột Train: ['Unnamed: 0', 'content', 'index_spans']
Cột Dev: ['Unnamed: 0', 'content', 'index_spans']
Cột Test đang sử dụng trong M3: ['content']


## File Test nào mới là file dùng để đánh giá cuối?

Trong thư mục sequence labeling có:

- `test_BIO_syllable.csv`
- `test_BIO_Word.csv`

Nhưng hai file này **chỉ dùng để tham khảo**.

File Test chính thức của ViHOS là:

`DATA/ViHOS_official/test/test.csv`

Trong M3, ta chỉ đọc cột `content` của file này và **không dùng `index_spans`**.

Lý do: nếu xem đáp án Test rồi dựa vào đó để sửa model, kết quả Test cuối cùng sẽ không còn phản ánh khả năng tổng quát hóa thật của model.


## Quy tắc chính thức của nhóm từ M3 trở đi

1. **Train**
   - Dùng để huấn luyện model.
   - Có thể xem và phân tích nhãn.

2. **Dev**
   - Không dùng làm dữ liệu huấn luyện chính.
   - Dùng để so sánh các phiên bản model và điều chỉnh lựa chọn.
   - Có thể xem lỗi và phân tích nhãn.

3. **Test**
   - Không dùng để chọn model.
   - Không dùng để sửa tham số hoặc thay đổi phương pháp.
   - Không phân tích đáp án trước khi model cuối cùng đã được chốt.
   - Chỉ dùng một lần ở giai đoạn đánh giá cuối.

4. **50 mẫu phân tích thủ công của đồ án**
   - Ưu tiên lấy từ Train và Dev.
   - Không lấy Test để tránh vô tình học từ đề thi cuối.

## Kết luận M3

ViHOS có:

- **8.844 Train**
- **1.106 Dev**
- **1.106 Test**

Từ sau M3, Test được coi là **sealed** cho tới giai đoạn đánh giá cuối.
